In [ ]:
%load_ext autoreload

In [ ]:
import altair as alt
import geopandas as gpd
import polars as pl
import polars.selectors as cs

%autoreload
from altair_utils import color_value_transit_teal
from downtown_travel_study_utils import parse_geog

In [ ]:
taz_analysis_neighborhoods_gis_filepath = r"Q:\GIS\Policy\San_Francisco\Analysis_Neighborhoods\taz2454-sf_only-with_analysis_neighborhoods.gpkg"
taz_analysis_neighborhoods = gpd.read_file(taz_analysis_neighborhoods_gis_filepath)[
    ["TAZ", "analysis_neighborhood"]
]

Caveats:

- delivery-food includes to any location in 2018, vs only to home in 2022
- survey instrument in 2018 is not explicit about whether all the other delivery\_\*
  variables include food deliveries, so we just assume they don't since delivery_food
  is a variable in itself
- for some reason the 2022 dataset distinguishes between packages and 'items'
- 2022 has a variable for 'Received packages at another location (e.g., Amazon Locker,
  package pick-up point)', whereas the corresponding variable for 2018 is just
  'Received package/delivery to a locker on travel date'. There's also a 'Received other
  type of delivery on travel date' in 2018 (though only 136 entries / 2226 person-days
  in the entire Bay Area). I decided to merge all of these into a single
  'delivery-other' variable.


In [ ]:
# don't re-calculate the full-week person-day weights because we did weights adjustment
# in the final step of the pipeline based on 3 day weights...
# def calc_fullweek_personday_weight(personday):
#     # recalculate person-day weights from the person weight and number of days in the
#     # person-day table
#     # not using the person-day weights as is from RSG or our pipeline because those are
#     # 3 day (Tue-Thu) weights and we want to capture deliveries outside of Tue-Thu too
#     return personday.with_columns(
#         personday_weight=(pl.col("psexpfac") / pl.len()).over("hhno", "pno")
#     )

In [ ]:
# geographies to use
# True: Downtown Today geographies (SF northeast core, rest of SF, rest of Bay Area)
# False: SF, rest of Bay ARea
geog_sf_ne_core = True

In [ ]:
personday_2018 = (
    (
        (  # get pdexpfac from pipeline results
            pl.read_csv(
                r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2018\Processing_20211018\v01\04-merge_skims\adj_weights\survey2018_pdayx_rewt_base2019.dat",
                separator=" ",
                columns=["hhno", "pno", "day", "pdexpfac"],
            ).with_columns(pl.col("pdexpfac").replace({"NA": 0}).cast(float))
        )
        .join(  # get delivery columns from raw
            pl.read_csv(
                r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2018\Deliverable_20211018\day.csv",
                columns=[
                    "hh_id",
                    "person_num",
                    "travel_date_dow",
                    "delivery_food",
                    "delivery_home",
                    "delivery_locker",
                    "delivery_work",
                    "delivery_other",
                    "delivery_none",
                ],
            ).rename({"hh_id": "hhno", "person_num": "pno", "travel_date_dow": "day"}),
            on=["hhno", "pno", "day"],
            how="left",
        )
        .join(
            parse_geog(
                pl.read_csv(
                    r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2018\Processing_20211018\v01\04-merge_skims\adj_weights\survey2018_precx_rewt_base2019.dat",
                    separator=" ",
                    columns=["hhno", "pno", "hhtaz", "pwtaz", "pagey"],
                ),
                taz_analysis_neighborhoods,
            ),
            on=["hhno", "pno"],
            how="left",  # keep persondays, just get hhtaz, pwtaz, age
        )
    )
    .filter(pl.col("pagey") > 17)  # just to be safe; 2019 survey is adults only
    .with_columns(
        cs.starts_with("delivery").replace_strict({1: True, 0: False, -9998: None})
    )
    .rename(  # rename for easier comparison between 2018/2022
        {
            "delivery_food": "delivery_any_food",
            # assume following is non-food but it's not explicit in survey instrument
            "delivery_home": "delivery_home_any",
            "delivery_locker": "delivery_locker_any",
            "delivery_work": "delivery_work_any",
            "delivery_other": "delivery_any_other",  # e.g. pick-up points?
            "delivery_none": "delivery_none",
        }
    )
    .select(  # rename to have uniform variable names between 2018/2022
        "hhno",
        "pno",
        "day",
        "hh_geog",
        "pw_geog",
        "pdexpfac",
        "delivery_none",
        delivery_food=pl.col("delivery_any_food"),  # includes any location!
        # assume following is non-food but it's not explicit in survey instrument
        delivery_home=pl.col("delivery_home_any"),
        delivery_work=pl.col("delivery_work_any"),
        delivery_other=(pl.col("delivery_locker_any") | pl.col("delivery_any_other")),
    )
)
personday_2018 = personday_2018.with_columns(
    # set persondays for which respondent says they got deliveries but not in any
    # delivery type (i.e. no delivery variable is True) to missing response / null
    # (this is still logically possible in 2018 given survey instrument wording,
    # but is impossible under 2022 survey instrument wording)
    **{
        col: pl.when(
            ~(
                pl.col("delivery_none")
                | pl.col("delivery_food")
                | pl.col("delivery_home")
                | pl.col("delivery_work")
                | pl.col("delivery_other")
            )
        )
        .then(pl.lit(None))
        .otherwise(pl.col(col))
        for col in personday_2018.select(cs.starts_with("delivery")).columns
    }
).with_columns(
    delivery_any=(~pl.col("delivery_none")),
    delivery_foodhomeother=(
        pl.col("delivery_food") | pl.col("delivery_home") | pl.col("delivery_other")
    ),
)
if not geog_sf_ne_core:
    personday_2018 = personday_2018.with_columns(
        pl.col("hh_geog").replace_strict(
            {
                "1. northeast core": "SF",
                "2. rest of SF": "SF",
                "3. rest of Bay Area": "rest of Bay Area",
            }
        )
    )

personday_2022 = (
    (
        (  # get pdexpfac from pipeline results
            pl.read_csv(
                r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241127\reformat_2019_rmoveonly\04-merge_skims\adj_weights\survey2023_pdayx_rewt_base2023.dat",
                separator=" ",
                columns=["hhno", "pno", "day", "pdexpfac"],
            )
        )
        .join(  # get delivery columns from raw
            pl.read_csv(
                r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Deliverable_20241127\day.csv",
                columns=[
                    "hh_id",
                    "person_num",
                    "travel_dow",
                    "delivery_2",
                    # drop 3: Someone came to do work at home (e.g., babysitter, housecleaning, lawn)
                    # "delivery_3",
                    "delivery_4",
                    "delivery_5",
                    "delivery_6",
                    "delivery_7",
                    "delivery_8",
                    "delivery_9",
                    "delivery_996",
                ],
            ).rename({"hh_id": "hhno", "person_num": "pno", "travel_dow": "day"}),
            on=["hhno", "pno", "day"],
            how="left",
        )
        .join(
            parse_geog(
                pl.read_csv(
                    r"Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241127\reformat_2019_rmoveonly\04-merge_skims\adj_weights\survey2023_precx_rewt_base2023.dat",
                    separator=" ",
                    columns=["hhno", "pno", "hhtaz", "pwtaz", "pagey"],
                ),
                taz_analysis_neighborhoods,
            ),
            on=["hhno", "pno"],
            how="left",  # keep persondays, just get hhtaz, pwtaz, age
        )
    )
    .filter(pl.col("pagey") > 17)  # adults only, to be consistent with 2019
    .with_columns(
        cs.starts_with("delivery").replace_strict({1: True, 0: False, 995: None})
    )
    .rename(
        {
            "delivery_2": "delivery_home_food",
            "delivery_4": "delivery_home_groceries",
            "delivery_5": "delivery_home_packages",
            "delivery_6": "delivery_work_personalpackages",
            # 7: # another location (e.g., Amazon Locker, package pick-up point)
            "delivery_7": "delivery_locker_packages",
            # 8: Other items delivered to home (e.g., appliance)
            "delivery_8": "delivery_home_other",  # exc packages
            "delivery_9": "delivery_work_other",  # exc personal packages
            "delivery_996": "delivery_none",
        }
    )
    .select(  # rename for easier comparison between 2018/2022
        "hhno",
        "pno",
        "day",
        "hh_geog",
        "pw_geog",
        "pdexpfac",
        "delivery_none",
        "delivery_home_food",
        "delivery_home_groceries",
        "delivery_home_packages",
        "delivery_work_personalpackages",
        "delivery_locker_packages",
        "delivery_home_other",
        "delivery_work_other",
    )
)
personday_2022 = personday_2022.with_columns(
    # set persondays for which respondent says they got deliveries but not in any
    # delivery type (i.e. no delivery variable is True) to missing response / null
    # (this is still logically possible in 2018 given survey instrument wording,
    # but is impossible under 2022 survey instrument wording)
    **{
        col: pl.when(
            ~(
                pl.col("delivery_none")
                | pl.col("delivery_home_food")
                | pl.col("delivery_home_groceries")
                | pl.col("delivery_home_packages")
                | pl.col("delivery_work_personalpackages")
                | pl.col("delivery_locker_packages")
                | pl.col("delivery_home_other")
                | pl.col("delivery_work_other")
            )
        )
        .then(pl.lit(None))
        .otherwise(pl.col(col))
        for col in personday_2022.select(cs.starts_with("delivery")).columns
    }
)
if not geog_sf_ne_core:
    personday_2022 = personday_2022.with_columns(
        pl.col("hh_geog").replace_strict(
            {
                "1. northeast core": "SF",
                "2. rest of SF": "SF",
                "3. rest of Bay Area": "rest of Bay Area",
            }
        )
    )

personday_2022_rough = (
    personday_2022.select(  # rename to have uniform variable names between 2018/2022
        "hhno",
        "pno",
        "day",
        "hh_geog",
        "pw_geog",
        "pdexpfac",
        "delivery_none",
        delivery_food=pl.col("delivery_home_food"),  # only includes to home!
        delivery_home=(
            pl.col("delivery_home_groceries")
            | pl.col("delivery_home_packages")
            | pl.col("delivery_home_other")
        ),
        delivery_work=(
            pl.col("delivery_work_personalpackages") | pl.col("delivery_work_other")
        ),
        # actually for all non work/home locations
        # "another location (e.g., Amazon Locker, package pick-up point)""
        delivery_other=pl.col("delivery_locker_packages"),
    ).with_columns(
        delivery_any=(~pl.col("delivery_none")),
        delivery_foodhomeother=(
            pl.col("delivery_food") | pl.col("delivery_home") | pl.col("delivery_other")
        ),
    )
)

personday_2022_fine = (
    personday_2022.select(
        "hhno",
        "pno",
        "day",
        "hh_geog",
        "pw_geog",
        "pdexpfac",
        "delivery_none",
        "delivery_home_groceries",
        "delivery_home_packages",
        "delivery_home_other",
        "delivery_work_personalpackages",
        "delivery_work_other",
        # decided to keep food under home for the 'fine' categories because we're not
        # doing the nested bar graph anyways
        "delivery_home_food",
        # delivery_food=pl.col("delivery_home_food"),  # only includes to home!
        delivery_other_packages=pl.col("delivery_locker_packages"),  # actually locker
        delivery_home_multicats=(
            pl.sum_horizontal(
                pl.col(
                    "delivery_home_food",
                    "delivery_home_groceries",
                    "delivery_home_packages",
                    "delivery_home_other",
                ).cast(int)
            )
            > 1
        ),
        delivery_home_2cats=pl.when(
            pl.sum_horizontal(
                pl.col(
                    "delivery_home_food",
                    "delivery_home_groceries",
                    "delivery_home_packages",
                    "delivery_home_other",
                ).cast(int)
            )
            == 2
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False)),
        delivery_home_3cats=pl.when(
            pl.sum_horizontal(
                pl.col(
                    "delivery_home_food",
                    "delivery_home_groceries",
                    "delivery_home_packages",
                    "delivery_home_other",
                ).cast(int)
            )
            == 3
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False)),
        delivery_home_4cats=pl.when(
            pl.sum_horizontal(
                pl.col(
                    "delivery_home_food",
                    "delivery_home_groceries",
                    "delivery_home_packages",
                    "delivery_home_other",
                ).cast(int)
            )
            == 4
        )
        .then(pl.lit(True))
        .otherwise(pl.lit(False)),
        delivery_work_2cats=(
            pl.col("delivery_work_personalpackages") & pl.col("delivery_work_other")
        ),
    )
    .with_columns(
        delivery_home_food=pl.when("delivery_home_multicats")
        .then(False)
        .otherwise(pl.col("delivery_home_food")),
        delivery_home_groceries=pl.when("delivery_home_multicats")
        .then(False)
        .otherwise(pl.col("delivery_home_groceries")),
        delivery_home_packages=pl.when("delivery_home_multicats")
        .then(False)
        .otherwise(pl.col("delivery_home_packages")),
        delivery_home_other=pl.when("delivery_home_multicats")
        .then(False)
        .otherwise(pl.col("delivery_home_other")),
        delivery_work_personalpackages=pl.when("delivery_work_2cats")
        .then(False)
        .otherwise(pl.col("delivery_work_personalpackages")),
        delivery_work_other=pl.when("delivery_work_2cats")
        .then(False)
        .otherwise(pl.col("delivery_work_other")),
    )
    .drop("delivery_home_multicats")
)

In [ ]:
print("make sure that when delivery-none is true, there are indeed no deliveries")
print(2018)
display(
    personday_2018.filter(pl.col("delivery_none"))
    .select(cs.starts_with("delivery"))
    .unique()
)
print(2022)
display(
    personday_2022.filter(pl.col("delivery_none"))
    .select(cs.starts_with("delivery"))
    .unique()
)

In [ ]:
def calc_sums(personday, geog):
    return (
        personday.with_columns(missing_response=pl.col("delivery_none").is_null())
        .group_by(geog)
        .agg(
            **{
                col: pl.col("pdexpfac").filter(col).sum()
                for col in (
                    personday.select(cs.starts_with("delivery")).columns
                    + ["missing_response"]
                )
            },
            total=pl.sum("pdexpfac"),
            total_exc_missingresponse=pl.col("pdexpfac")
            .filter(~pl.col("missing_response"))
            .sum(),
        )
        .sort(geog)
    )

In [ ]:
def add_shares(df, col, sum_over_cols="year"):
    return df.with_columns(
        (pl.col("weighted") / pl.col("weighted").sum().over(sum_over_cols)).alias(
            f"{col}-share"
        )
    )


def plot_dist(df, df_name, col):
    # df = (
    #     df.group_by(col).agg(weighted=pl.sum("psexpfac"), unweighted=pl.len()).sort(col)
    # )
    df = df.with_columns(pl.col("delivery_type").replace({"missing_response": None}))
    return df, alt.Chart(df.drop_nulls()).mark_bar().encode(
        x=alt.X(f"{col}:O", title=None, axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("weighted", title=col),
        color=color_value_transit_teal,
    ).properties(title=f"{df_name}")
    # + df.plot.bar(
    #     x=col, y="unweighted", title=f"{df_name}\n{col}: unweighted", frame_width=200
    # ).opts(xrotation=45)


def plot_dists(df2018, df2022, df_name, col):
    df2018, plot2018 = plot_dist(df2018, f"{df_name} preCOVID", col)
    df2022, plot2022 = plot_dist(df2022, f"{df_name} postCOVID", col)
    display(df2018)
    display(df2022)
    df = pl.concat(
        [
            df2018.with_columns(year=pl.lit(2019)),
            df2022.with_columns(year=pl.lit(2023)),
        ]
    )
    print("look here to get null shares (non-response rate):")
    display(add_shares(df.drop_nulls(), col))
    df = df.drop_nulls()
    print("sums 2018/2022", df2018.sum(), df2022.sum())
    return df, (plot2018 | plot2022).resolve_scale(y="shared")

In [ ]:
# home location
delivery_home_2018_wide = calc_sums(personday_2018, "hh_geog")
delivery_home_2022_rough_wide = calc_sums(personday_2022_rough, "hh_geog")
delivery_home_2022_fine_wide = calc_sums(personday_2022_fine, "hh_geog")
# work location
delivery_work_2018_wide = calc_sums(personday_2018, "pw_geog")
delivery_work_2022_rough_wide = calc_sums(personday_2022_rough, "pw_geog")
delivery_work_2022_fine_wide = calc_sums(personday_2022_fine, "pw_geog")

In [ ]:
delivery_home_2018_wide.unpivot(
    index=["hh_geog"],
    on=[
        "delivery_none",
        "delivery_any",
        "delivery_food",
        "delivery_home",
        # Downtown Today report not interested in deliveries to work location
        # "delivery_work",
        "delivery_other",
        # drop missing responses to calculate per personday rates
        # "missing_response",
        "total_exc_missingresponse",
    ],
    variable_name="delivery_type",
    value_name="weighted",
)


In [ ]:
delivery_home_2022_rough_wide.select("hh_geog", "total_exc_missingresponse")

In [ ]:
delivery_home_2018_long = (
    delivery_home_2018_wide.unpivot(
        index=["hh_geog"],
        on=[
            "delivery_none",
            "delivery_any",
            "delivery_food",
            "delivery_home",
            # Downtown Today report not interested in deliveries to work location
            # "delivery_work",
            "delivery_other",
            # drop missing responses to calculate per personday rates
            # "missing_response",
        ],
        variable_name="delivery_type",
        value_name="weighted",
    )
    .join(
        delivery_home_2018_wide.select("hh_geog", "total_exc_missingresponse"),
        on="hh_geog",
    )
    .with_columns(
        weighted_per_hhgeog_personday=(
            pl.col("weighted") / pl.col("total_exc_missingresponse")
        )
    )
    .drop("total_exc_missingresponse")
)
delivery_home_2022_rough_long = (
    delivery_home_2022_rough_wide.unpivot(
        index=["hh_geog"],
        on=[
            "delivery_none",
            "delivery_any",
            "delivery_food",
            "delivery_home",
            # Downtown Today report not interested in deliveries to work location
            # "delivery_work",
            "delivery_other",
            # drop missing responses to calculate per personday rates
            # "missing_response",
        ],
        variable_name="delivery_type",
        value_name="weighted",
    )
    .join(
        delivery_home_2022_rough_wide.select("hh_geog", "total_exc_missingresponse"),
        on="hh_geog",
    )
    .with_columns(
        weighted_per_hhgeog_personday=(
            pl.col("weighted") / pl.col("total_exc_missingresponse")
        )
    )
    .drop("total_exc_missingresponse")
)
delivery_home_2022_fine_long = add_shares(
    (
        delivery_home_2022_fine_wide.unpivot(
            index=["hh_geog"],
            on=[
                "delivery_none",
                "delivery_other_packages",
                "delivery_home_food",
                "delivery_home_groceries",
                "delivery_home_packages",
                "delivery_home_other",
                "delivery_home_2cats",
                "delivery_home_3cats",
                "delivery_home_4cats",
                # Downtown Today report not interested in deliveries to work location
                # "delivery_work_personalpackages",
                # "delivery_work_other",
                # "delivery_work_2cats",
                # drop missing responses to calculate per personday rates
                # and for calculating shares
                # "missing_response",
            ],
            variable_name="delivery_type",
            value_name="weighted",
        )
        .with_columns(year=pl.lit(2023))
        .join(
            delivery_home_2022_rough_wide.select(
                "hh_geog", "total_exc_missingresponse"
            ),
            on="hh_geog",
        )
        .with_columns(
            weighted_per_hhgeog_personday=(
                pl.col("weighted") / pl.col("total_exc_missingresponse")
            )
        )
        .drop("total_exc_missingresponse")
        .with_columns(
            delivery_type_rough=pl.col("delivery_type").str.split("_").list.get(1)
        )
    ),
    "delivery_type",
    sum_over_cols=["year", "hh_geog", "delivery_type_rough"],
).sort("hh_geog", "delivery_type")


In [ ]:
# # work deliveries by work location
# delivery_work_2018_long = delivery_work_2018_wide.unpivot(
#     index=["pw_geog"],
#     on=[
#         "delivery_work",
#     ],
#     variable_name="delivery_type",
#     value_name="weighted",
# )
# delivery_work_2022_rough_long = delivery_work_2022_rough_wide.unpivot(
#     index=["pw_geog"],
#     on=[
#         "delivery_work",
#     ],
#     variable_name="delivery_type",
#     value_name="weighted",
# )
# delivery_work_2022_fine_long = add_shares(
#     delivery_work_2022_fine_wide.unpivot(
#         index=["pw_geog"],
#         on=["delivery_work_personalpackages", "delivery_work_other"],
#         variable_name="delivery_type",
#         value_name="weighted",
#     ).with_columns(year=pl.lit(2023)),
#     "delivery_type",
#     sum_over_cols=["year", "pw_geog"],
# ).sort("pw_geog", "delivery_type")




In [ ]:
# delivery_home_2018_wide.write_csv("output/delivery-home-2018-wide.csv")
# delivery_home_2022_rough_wide.write_csv("output/delivery-home-2022-rough-wide.csv")
# delivery_home_2022_fine_wide.write_csv("output/delivery-home-2022-fine-wide.csv")
delivery_home_2022_fine_long.write_csv("output/delivery-home-2022-fine-long.csv")

In [ ]:
delivery_home_necore_long, _ = plot_dists(
    delivery_home_2018_long.filter(pl.col("hh_geog") == "1. northeast core"),
    delivery_home_2022_rough_long.filter(pl.col("hh_geog") == "1. northeast core"),
    "persons living in NE core (delivery type can overlap)",
    "delivery_type",
)
delivery_home_necore_chart = (
    alt.Chart(delivery_home_necore_long)
    .mark_bar()
    .encode(
        x="year:O",
        y="weighted_per_hhgeog_personday",
        color=color_value_transit_teal,
        column=alt.Column("delivery_type", header=alt.Header(labelAngle=-20)),
    )
    .properties(title="hh geog NE core (delivery type can overlap)")
)
delivery_home_necore_long.write_csv("output/delivery-home-necore-long.csv")
delivery_home_necore_chart

In [ ]:
delivery_home_restofsf_long, _ = plot_dists(
    delivery_home_2018_long.filter(pl.col("hh_geog") == "2. rest of SF"),
    delivery_home_2022_rough_long.filter(pl.col("hh_geog") == "2. rest of SF"),
    "persons living in rest of SF (delivery type can overlap)",
    "delivery_type",
)
delivery_home_restofsf_chart = (
    alt.Chart(delivery_home_restofsf_long)
    .mark_bar()
    .encode(
        x="year:O",
        y="weighted_per_hhgeog_personday",
        color=color_value_transit_teal,
        column=alt.Column("delivery_type", header=alt.Header(labelAngle=-20)),
    )
    .properties(title="hh geog rest of SF")
)
delivery_home_restofsf_long.write_csv("output/delivery-home-restofsf-long.csv")
delivery_home_restofsf_chart

In [ ]:
delivery_home_restofbayarea_long, _ = plot_dists(
    delivery_home_2018_long.filter(pl.col("hh_geog") == "3. rest of Bay Area"),
    delivery_home_2022_rough_long.filter(pl.col("hh_geog") == "3. rest of Bay Area"),
    "persons living in rest of SF (delivery type can overlap)",
    "delivery_type",
)
delivery_home_restofbayarea_chart = (
    alt.Chart(delivery_home_restofbayarea_long)
    .mark_bar()
    .encode(
        x="year:O",
        y="weighted_per_hhgeog_personday",
        color=color_value_transit_teal,
        column=alt.Column("delivery_type", header=alt.Header(labelAngle=-20)),
    )
    .properties(title="hh geog rest of Bay Area")
)
delivery_home_restofbayarea_long.write_csv(
    "output/delivery-home-restofbayarea-long.csv"
)
delivery_home_restofbayarea_chart

In [ ]:
(
    alt.Chart(
        delivery_home_2022_fine_long.filter(pl.col("delivery_type_rough") == "home")
    )
    .mark_bar()
    .encode(
        # x="year:O",
        y="delivery_type-share",
        color="delivery_type",
        # color=color_value_transit_teal,
        column=alt.Column("hh_geog", header=alt.Header(labelAngle=-20)),
    )
    .properties(title="hh geog rest of Bay Area")
)


In [ ]:
# # by work location
# delivery_work_2018_long.write_csv("output/delivery-work-2018-long.csv")
# delivery_work_2022_fine_long.write_csv("output/delivery-work-2022-fine-long.csv")


In [ ]:
# # work deliveries by work geography
# delivery_work_necore_df, _ = plot_dists(
#     delivery_work_2018_long.filter(pl.col("pw_geog") == "1. northeast core"),
#     delivery_work_2022_rough_long.filter(pl.col("pw_geog") == "1. northeast core"),
#     "persons working in NE core (delivery type can overlap)",
#     "delivery_type",
# )
# delivery_work_restofsf_df, _ = plot_dists(
#     delivery_work_2018_long.filter(pl.col("pw_geog") == "2. rest of SF"),
#     delivery_work_2022_rough_long.filter(pl.col("pw_geog") == "2. rest of SF"),
#     "persons living in rest of SF (delivery type can overlap)",
#     "delivery_type",
# )
# delivery_work_restofbayarea_df, _ = plot_dists(
#     delivery_work_2018_long.filter(pl.col("pw_geog") == "3. rest of Bay Area"),
#     delivery_work_2022_rough_long.filter(pl.col("pw_geog") == "3. rest of Bay Area"),
#     "persons living in rest of SF (delivery type can overlap)",
#     "delivery_type",
# )

# (
#     alt.Chart(delivery_work_necore_df)
#     .mark_bar()
#     .encode(
#         x="year:O",
#         y="weighted",
#         color=color_value_transit_teal,
#         column=alt.Column("delivery_type", header=alt.Header(labelAngle=-20)),
#     )
#     .properties(title="pw geog NE core")
#     | alt.Chart(delivery_work_restofsf_df)
#     .mark_bar()
#     .encode(
#         x="year:O",
#         y="weighted",
#         color=color_value_transit_teal,
#         column=alt.Column("delivery_type", header=alt.Header(labelAngle=-20)),
#     )
#     .properties(title="pw geog rest of SF")
#     | alt.Chart(delivery_work_restofbayarea_df)
#     .mark_bar()
#     .encode(
#         x="year:O",
#         y="weighted",
#         color=color_value_transit_teal,
#         column=alt.Column("delivery_type", header=alt.Header(labelAngle=-20)),
#     )
#     .properties(title="pw geog rest pf Bay Area")
# )